# Journey A — Admin สร้างหน่วยงานและผู้ใช้ (ยิง API ทีละขั้น พร้อมอีเมลจริง)

สเปกกำหนดว่าขั้นตอนนี้ **ไม่มีหน้าจอ** — แอดมินเรียก API ตรง ๆ notebook นี้จึงเดินแทนหน้าจอ
โดยยิงทีละ endpoint ให้เห็นทั้ง request และ response ตอนสาธิต

| ขั้น | ใครทำ | endpoint |
| --- | --- | --- |
| 0.6 | Admin | `POST /api/admin/organizations` — สร้างหน่วยงานล่วงหน้า (ไม่บังคับ) เก็บ `organizationId` ไว้ผูกคำเชิญ |
| 1 | Admin | `POST /api/admin/invitations` — สร้างบัญชี `PENDING` + ออก activation key แล้ว **ส่งอีเมล** |
| 2 | ผู้ถูกเชิญ | `GET /api/auth/invitation?token=` — ตรวจว่าลิงก์ยังใช้ได้ |
| 3 | ผู้ถูกเชิญ | `POST /api/auth/thaid/start` → ยืนยันบนเว็บ ThaiD — เทียบเลขบัตรกับที่บันทึกไว้ |
| 4 | ผู้ถูกเชิญ | `POST /api/auth/activate` — ตั้งรหัสผ่าน · บัญชีเป็น `ACTIVE` · ผูก role · ปิด key |
| 5 | ผู้ถูกเชิญ | `GET /api/auth/me` — ดูว่าได้ role และหน่วยงานอะไร |

## เปลี่ยนอะไรไปบ้างหลังย้ายสคีมา (2026-08-12)

ถ้าเคยรัน notebook นี้แล้วตอนนี้ขึ้น `400 organizationId` — เพราะสามเรื่องนี้

1. **`Invitation` กลายเป็น `iam.activation_key`** ลำดับใหม่คือสร้าง `user_account` เป็น `PENDING` ก่อน
   แล้วค่อยออก key ให้บัญชีนั้น (ไม่ใช่ผูกคำเชิญไว้กับอีเมลลอย ๆ) · คำตอบจึงคืน
   `activationKeyId` กับ `userAccountId` ไม่ใช่ `invitationId`
2. **role ที่ผูกกับหน่วยงานต้องระบุ `organizationId`** — `ORGANIZATION_USER` และ
   `ORGANIZATION_APPROVER` ส่วน role ฝั่ง BDI ไม่ต้องส่ง ระบบผูกกับหน่วยงาน BDI ให้เอง
3. **รหัส role เปลี่ยนสองตัวและเพิ่มใหม่สองตัว** ของเดิม `BDI_APPROVER` / `BDI_SPECIALIST`
   ใช้ไม่ได้แล้ว

> ⚠️ **หนึ่งหน่วยงานมี `ORGANIZATION_USER` ที่ ACTIVE ได้คนเดียว** (เช่นเดียวกับ
> `ORGANIZATION_APPROVER`) เชิญคนใหม่เข้าหน่วยงานที่มีอยู่แล้ว **คนเดิมจะถูกเพิกถอนสิทธิ์ทันที**
> ที่คนใหม่ยืนยัน OTP เสร็จ · ถ้าไม่อยากทับของเดิม ให้เลือกหน่วยงานอื่นหรือเชิญเป็น role ฝั่ง BDI
> กฎนี้มาจาก sheet `user_account` ใน Excel — ดู `CLAUDE.md` หัวข้อ Auth

## ThaiD ต้องใช้ credentials ที่ตรงกับ checkout

ขั้นที่ 3 ยิง ThaiD ของจริงเสมอ ไม่มีโหมดข้าม (โหมด `THAID_BYPASS` ถูกถอดออกเมื่อ 2026-09-02
พร้อมกับที่กรมการปกครองเปิด redirect URI ของโดเมนจริงให้)

- **บน `main`** ใช้ credentials ของโครงการได้เลย ทะเบียนผูกไว้กับ
  `https://bdi.thammasorn.org/auth/callback/thaid`
- **บน dev checkout** ต้องใช้ **client ตัวอย่างของ sandbox**
  (`../assets/thaid/thaid sandbox.postman_environment.json`) ซึ่งรับ `redirect_uri` อะไรก็ได้ —
  credentials ของโครงการจะถูกปฏิเสธตั้งแต่ขั้น authorize เพราะ `localhost` ไม่อยู่ในทะเบียนแล้ว
  ดู `../docs/07-thaid-integration.md` §4.1

## ก่อนเริ่ม

- ต้องมี `requests` (`python3 -m pip install requests`) และเคอร์เนล Python ที่รัน notebook ได้
  เครื่องนี้ยังไม่มี jupyter — เปิดไฟล์นี้ใน VS Code (ต้องมี `ipykernel`) หรือ
  `python3 -m pip install jupyterlab ipykernel` แล้ว `jupyter lab`
- **อีเมลจะถูกส่งจริงก็ต่อเมื่อ checkout นั้นตั้ง `SMTP_USER` / `SMTP_PASS` ไว้แล้ว**
  ตอนนี้มีแค่ `main` ที่ตั้งไว้ — checkout อื่นจะพิมพ์อีเมลลง log แทน (เซลล์ที่ 2 บอกให้ว่าอยู่โหมดไหน)
- ใส่อีเมลของตัวเองในเซลล์ถัดไป ถ้าใช้ Gmail แนะนำ **plus-addressing**
  (`ชื่อคุณ+demo1@gmail.com`) จะเชิญคนใหม่กี่รอบก็ได้โดยไม่ชนบัญชีเดิม เมลเข้ากล่องเดียวกัน

รายละเอียดของแต่ละหน้าจอที่คู่กับ API เหล่านี้อยู่ใน [`../docs/03-demo-walkthrough.md`](../docs/03-demo-walkthrough.md)


In [9]:
import json
import subprocess
from pathlib import Path
from urllib.parse import urlparse, parse_qs
import uuid
import requests

import random

def generate_thai_id():
    # สุ่ม 12 หลักแรก (ให้หลักแรกเป็น 1-8)
    first_digit = str(random.randint(1, 8))
    rest_11_digits = ''.join([str(random.randint(0, 9)) for _ in range(11)])
    
    twelve_digits = first_digit + rest_11_digits
    
    # คำนวณหลักที่ 13 (Checksum)
    total_sum = 0
    for i, digit in enumerate(twelve_digits):
        weight = 13 - i
        total_sum += int(digit) * weight
        
    mod = total_sum % 11
    checksum = (11 - mod) % 10
    
    return twelve_digits + str(checksum)

print(generate_thai_id()) # ตัวอย่างผลลัพธ์: "8765432109874"

# ── แก้ตรงนี้ ────────────────────────────────────────────────────────────
CHECKOUT = Path("/hdd1tb/bdi-project/main")     # checkout ที่จะยิง API ใส่ (main = ส่งอีเมลจริง)
API = "https://bdi-api.thammasorn.org"          # main
                                                # dev_20260812_update-home-page http://localhost:4140

INVITE_EMAIL = "thammasorn.han+demo21@gmail.com"

# ORGANIZATION_USER · ORGANIZATION_APPROVER   ← ใส่ ORGANIZATION_ID หรือเว้นว่าง
# BDI_OFFICER · BDI_DATASET_SPECIALIST · BDI_FINAL_APPROVER · BDI_LEGAL_OFFICER
# SYSTEM_ADMINISTRATOR                        ← ไม่ต้องมี ผูกกับหน่วยงาน BDI ให้เอง
ROLE = "ORGANIZATION_USER"

# เว้นว่าง = เชิญคนที่จะมา "สร้างหน่วยงานของตัวเอง" (จุดเริ่มของ Journey B)
# ระบบจะเตรียมหน่วยงานเปล่ากับคำขอฉบับร่างรอไว้ให้ · ใส่ id = เข้าหน่วยงานที่มีอยู่แล้ว
ORGANIZATION_ID = ""   # เซลล์ถัดไปลิสต์หน่วยงานที่เลือกได้
CID = generate_thai_id()  # เลขบัตร 13 หลัก **บังคับทุก role** และตรวจ checksum
                       # ค่านี้คือเลขตั้งต้นของ ThaiD sandbox จึงใช้สาธิตได้ทันที

# สร้างหน่วยงานใหม่ผ่าน API ในเซลล์ 0.6 หรือไม่ (เฉพาะ role ฝั่งหน่วยงาน)
#   True  → ยิง POST /api/admin/organizations ด้วย ORG_FIELDS แล้วตั้ง ORGANIZATION_ID ให้เอง
#   False → ใช้ ORGANIZATION_ID ข้างบน (หรือเว้นว่างให้ระบบสร้างหน่วยงานเปล่าตอนเชิญ)
CREATE_ORG = True
ORG_FIELDS = {
    "organizationCode": f"MOT-DEMO-{str(uuid.uuid4())[:6]}",   # ห้ามซ้ำกับหน่วยงานอื่น — ซ้ำได้ 409 code_exists
    "nameTh": "สำนักงานปลัดกระทรวงสาสุกเดโม่",
    "nameEn": "Office of the Permanent Secretary, Ministry of Transport, Demo",
    "organizationType": "ส่วนราชการ",
    # ที่อยู่ส่งเป็น "ชื่อ" ไม่ใช่รหัส และต้องสะกดตรงกับในระบบ ("ดุสิต" ไม่ใช่ "เขตดุสิต")
    "addressLine": "38 ถนนราชดำเนินนอก",
    "province": "กรุงเทพมหานคร",
    "district": "ป้อมปราบศัตรูพ่าย",
    "subdistrict": "วัดโสมนัส",
    "postalCode": "10100",               # เว้นได้ถ้าใส่ตำบลครบ ระบบเติมให้
    "phone": "022835000",
    "email": "saraban@motdemo.go.th",
    "websiteUrl": "https://www.mot.go.th",
}
# ─────────────────────────────────────────────────────────────────────────

ORG_SCOPED_ROLES = {"ORGANIZATION_USER", "ORGANIZATION_APPROVER"}

# main ตั้ง APP_URL เป็น https จึงออก session cookie แบบ Secure — ยิงผ่าน
# http://localhost:4000 จะล็อกอินผ่านแต่ทุก call ถัดไปได้ 401 เพราะ cookie ไม่ถูกส่งกลับ
# ใช้ URL https ข้างบนกับ main เสมอ


def env(name: str, default: str = "") -> str:
    """อ่านค่าจาก .env ของ checkout — ไฟล์นี้ไม่อยู่ใน git และเก็บ credential จริง"""
    for line in (CHECKOUT / ".env").read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        if key.strip() == name:
            return value.strip().strip('"').strip("'")
    return default


ADMIN_TOKEN = env("ADMIN_API_TOKEN")
APP_URL = env("APP_URL", "http://localhost:3000")
SMTP_USER = env("SMTP_USER")

http = requests.Session()  # เก็บ session cookie ให้อัตโนมัติหลังยืนยัน OTP


def call(method: str, path: str, **kwargs):
    """ยิง API แล้วพิมพ์ทั้งสถานะและ body ให้ดูสด ๆ ตอนสาธิต"""
    response = http.request(method, API + path, timeout=30, **kwargs)
    print(f"{method} {path}  →  {response.status_code} {response.reason}\n")
    try:
        body = response.json()
    except ValueError:
        print(response.text[:800])
        return response, None
    print(json.dumps(body, ensure_ascii=False, indent=2))
    return response, body


def psql(sql: str) -> str:
    """
    ถามฐานข้อมูลของ checkout ตรง ๆ

    ใช้เพราะยังไม่มี endpoint ที่ให้แอดมินลิสต์หน่วยงานได้ด้วย x-admin-token
    (ทุก endpoint ของหน่วยงานต้องมี session) รันได้เฉพาะกับ checkout บนเครื่องนี้
    """
    out = subprocess.run(
        ["docker", "compose", "exec", "-T", "postgres", "psql", "-U", "bdi", "-d", "bdi", "-Atc", sql],
        cwd=CHECKOUT, capture_output=True, text=True,
    )
    return out.stdout.strip() if out.returncode == 0 else f"(psql ล้มเหลว: {out.stderr.strip()[:200]})"


print("พร้อมแล้ว — ยิงไปที่", API)


5778861361247
พร้อมแล้ว — ยิงไปที่ https://bdi-api.thammasorn.org


## 0. ตรวจก่อนว่าระบบพร้อม และอีเมลจะถูกส่งจริงหรือไม่

In [ ]:
call("GET", "/health/ready")

print()
if SMTP_USER:
    print(f"โหมดอีเมล : ส่งจริง ผ่าน {SMTP_USER}")
else:
    print("โหมดอีเมล : พิมพ์ลง log เท่านั้น (SMTP_USER ว่าง)")
    print("            ถ้าต้องการอีเมลจริง ตั้ง SMTP_USER/SMTP_PASS ใน .env แล้ว")
    print("            docker compose up -d backend  —  หรือชี้ CHECKOUT ไปที่ main")

print("ลิงก์ในอีเมลจะชี้ไปที่ :", APP_URL)
print("ADMIN_API_TOKEN       :", "อ่านจาก .env ได้แล้ว" if ADMIN_TOKEN else "ไม่พบใน .env")
print("อีเมลที่จะเชิญ         :", INVITE_EMAIL)
print("role                  :", ROLE)

## 0.5 เลือกหน่วยงาน (เฉพาะ role ฝั่งหน่วยงาน)

`ORGANIZATION_USER` และ `ORGANIZATION_APPROVER` เลือกได้สองแบบ:

- **เว้น `ORGANIZATION_ID` ว่าง** — คนนี้จะเป็นผู้ก่อตั้งหน่วยงานใหม่ ระบบสร้างหน่วยงาน
  เปล่าสถานะ `PENDING_REGISTRATION` พร้อมคำขอฉบับร่างรอไว้ พอเขาล็อกอินแล้วกด
  *สร้างหน่วยงาน* จะเจอร่างใบนั้น นี่คือจุดเริ่มต้นของ Journey B
- **ใส่ id จากรายการข้างล่าง** — เข้าไปเป็นสมาชิกของหน่วยงานที่เปิดใช้งานแล้ว

คอลัมน์ท้ายบอกว่าหน่วยงานนั้น**มีใครถือ role นี้อยู่แล้วหรือยัง** ถ้ามี คนเดิมจะถูกเพิกถอน
เมื่อคนใหม่ยืนยัน OTP เสร็จ — ดีไซน์กำหนดให้หนึ่งหน่วยงานมีผู้ถือ role นี้ได้ทีละคน

> role ฝั่ง BDI ข้ามเซลล์นี้ไปได้เลย

In [11]:
if ROLE in ORG_SCOPED_ROLES:
    rows = psql(
        "select o.id, o.name_th, coalesce(string_agg(u.email, ', '), '—') "
        "from organization.organization o "
        "left join iam.user_role_assignment a on a.organization_id = o.id "
        "  and a.status = 'ACTIVE' "
        "  and a.role_id = (select id from iam.role where code = '" + ROLE + "') "
        "left join iam.user_account u on u.id = a.user_account_id "
        "where o.status = 'ACTIVE' and o.organization_code <> 'BDI' "
        "group by o.id, o.name_th order by o.name_th"
    )
    print(f"หน่วยงานที่ ACTIVE — คอลัมน์ท้ายคือผู้ที่ถือ {ROLE} อยู่ตอนนี้\n")
    for row in rows.splitlines():
        if "|" in row:
            oid, name, holder = row.split("|", 2)
            print(f"  {oid}  {name}")
            print(f"  {'':36}  ปัจจุบัน: {holder}\n")
    print("คัดลอก id ไปใส่ ORGANIZATION_ID ในเซลล์ที่ 1 แล้วรันเซลล์นั้นใหม่")
    print("หรือปล่อยว่างไว้ ถ้าอยากให้คนนี้เป็นผู้ก่อตั้งหน่วยงานใหม่")
else:
    print(f"{ROLE} เป็น role ฝั่ง BDI — ไม่ต้องระบุหน่วยงาน ระบบผูกกับหน่วยงาน BDI ให้เอง")

หน่วยงานที่ ACTIVE — คอลัมน์ท้ายคือผู้ที่ถือ ORGANIZATION_USER อยู่ตอนนี้

  302fadad-995a-429d-bb21-94dc0287a9ad  test-0001
                                        ปัจจุบัน: nj.nonreason@gmail.com

  71656feb-fd59-4015-b48f-cb892c75754b  test-0010
                                        ปัจจุบัน: amorn.chok22+org_user@gmail.com

  252ffcda-2e9d-45fb-8cae-07467ea56f01  ทดสอบ Organization Name
                                        ปัจจุบัน: thanat.lap+demo3@gmail.com

  17901807-5754-4664-9411-5d7159642524  สำนักงานปลัดกระทรวงคมนาคมเดโม่
                                        ปัจจุบัน: thammasorn.han+demo4@gmail.com

  9053091a-fa1a-4c38-b3c9-0b2dcae19a41  สำนักงานสถิติแห่งชาติ
                                        ปัจจุบัน: user@nso.go.th

คัดลอก id ไปใส่ ORGANIZATION_ID ในเซลล์ที่ 1 แล้วรันเซลล์นั้นใหม่
หรือปล่อยว่างไว้ ถ้าอยากให้คนนี้เป็นผู้ก่อตั้งหน่วยงานใหม่


## 0.6 สร้างหน่วยงานล่วงหน้าผ่าน API (ไม่บังคับ)

อีกทางเลือกหนึ่งของเซลล์ก่อนหน้า แทนที่จะเลือกหน่วยงานที่มีอยู่ ให้ **สร้างหน่วยงานใหม่**
ด้วย `POST /api/admin/organizations` แล้วผูกคำเชิญกับหน่วยงานนั้น ข้อมูลที่กรอกไว้ที่นี่จะถูก
**เติมให้อัตโนมัติ** ในแบบฟอร์มของผู้ใช้ตอนเขาเริ่มลงทะเบียน (Journey B)

- ตั้ง `CREATE_ORG = True` ในเซลล์ที่ 1 เพื่อรันขั้นนี้ (เฉพาะ role ฝั่งหน่วยงาน)
- บังคับแค่ `organizationCode` กับ `nameTh` ที่เหลือเว้นได้ · ที่อยู่ส่งเป็น "ชื่อ" ต้องสะกดตรง
- รหัสหน่วยงานซ้ำ → `409 code_exists` · ชื่ออำเภอ/ตำบลผิด → `400` พร้อมบอกช่องที่ผิด

หน่วยงานที่สร้างจะเป็นสถานะ `PENDING_REGISTRATION` และจะ `ACTIVE` ก็ต่อเมื่อผ่าน Journey B ครบ

In [12]:
if ROLE not in ORG_SCOPED_ROLES:
    print(f"{ROLE} เป็น role ฝั่ง BDI — ไม่ต้องมีหน่วยงาน ข้ามเซลล์นี้")
elif not CREATE_ORG:
    print("CREATE_ORG = False — ไม่ได้สร้างหน่วยงานใหม่")
    print("ใช้ ORGANIZATION_ID ที่ตั้งไว้ในเซลล์ที่ 1:",
          ORGANIZATION_ID or "(ว่าง = ให้ระบบสร้างหน่วยงานเปล่าตอนเชิญ)")
else:
    response, org = call(
        "POST",
        "/api/admin/organizations",
        headers={"x-admin-token": ADMIN_TOKEN},
        json=ORG_FIELDS,
    )
    if response.status_code == 201:
        ORGANIZATION_ID = org["organization"]["id"]
        print("\n✅ สร้างหน่วยงานแล้ว")
        print("   organizationId =", ORGANIZATION_ID, "  (เซลล์เชิญข้อ 1 จะผูกคำเชิญให้เอง)")
        print("   สถานะ", org["organization"]["status"], "— จะ ACTIVE เมื่อผ่าน Journey B ครบ")
    elif response.status_code == 409:
        # รหัสนี้มีหน่วยงานอยู่แล้ว (เช่นรันเซลล์นี้ซ้ำ) — ใช้หน่วยงานเดิมต่อ
        # ถ้าปล่อย ORGANIZATION_ID ว่าง เซลล์เชิญจะส่งคำเชิญแบบไม่ผูกหน่วยงาน ระบบจะสร้าง
        # หน่วยงานเปล่าให้แทน แล้วฟอร์มของผู้ใช้จะไม่มีข้อมูล prefill (อาการที่เจอบ่อย)
        ORGANIZATION_ID = org["organizationId"] if org else ""
        print("\nℹ️  รหัสหน่วยงานนี้มีอยู่แล้ว — จะใช้หน่วยงานเดิมต่อ")
        print("   organizationId =", ORGANIZATION_ID, "  (เซลล์เชิญข้อ 1 จะผูกคำเชิญกับหน่วยงานนี้)")
        print("   ถ้าตั้งใจสร้างหน่วยงาน 'ใหม่' ให้เปลี่ยน organizationCode ใน ORG_FIELDS แล้วรันใหม่")
    elif response.status_code == 400:
        print("\nข้อมูลไม่ผ่าน — ดู fields ข้างบนว่าช่องไหน (ชื่ออำเภอ/ตำบลต้องสะกดตรงกับในระบบ)")

POST /api/admin/organizations  →  201 Created

{
  "organization": {
    "id": "759f8e08-07e9-4ae6-9ada-3c9aa934d199",
    "organizationCode": "MOT-DEMO-2b192c",
    "organizationType": "ส่วนราชการ",
    "nameTh": "สำนักงานปลัดกระทรวงสาสุกเดโม่",
    "nameEn": "Office of the Permanent Secretary, Ministry of Transport, Demo",
    "status": "PENDING_REGISTRATION",
    "addressLine": "38 ถนนราชดำเนินนอก",
    "road": null,
    "province": "กรุงเทพมหานคร",
    "district": "ป้อมปราบศัตรูพ่าย",
    "subdistrict": "วัดโสมนัส",
    "postalCode": "10100",
    "phone": "022835000",
    "email": "saraban@motdemo.go.th",
    "websiteUrl": "https://www.mot.go.th",
    "parentOrganizationId": null,
    "activatedAt": null,
    "createdAt": "2026-08-29T16:25:11.503Z"
  }
}

✅ สร้างหน่วยงานแล้ว
   organizationId = 759f8e08-07e9-4ae6-9ada-3c9aa934d199   (เซลล์เชิญข้อ 1 จะผูกคำเชิญให้เอง)
   สถานะ PENDING_REGISTRATION — จะ ACTIVE เมื่อผ่าน Journey B ครบ


## 1. Admin ส่งคำเชิญ

ป้องกันด้วย `x-admin-token` ไม่ใช่ session เพราะผู้เรียกคือสคริปต์ของผู้ดูแลระบบ ไม่ใช่คนที่ล็อกอิน

ระบบจะ **สร้างบัญชีเป็น `PENDING`** ก่อน แล้วออก activation key ให้บัญชีนั้น
คีย์ถูกเก็บเป็น HMAC-SHA-256 **คำตอบจึงไม่คืนตัวคีย์กลับมา** — มีอยู่ในอีเมลเท่านั้น
ส่งลิงก์ซ้ำให้คนเดิมด้วย `POST /api/admin/invitations/{id}/resend` (ไม่รับ body) คีย์เก่าจะถูก revoke อัตโนมัติ เหลือลิงก์ที่ใช้ได้อันเดียวเสมอ — **เชิญอีเมลเดิมซ้ำผ่าน POST /invitations ไม่ได้แล้ว จะได้ 409**

In [13]:
ORGANIZATION_ID

'759f8e08-07e9-4ae6-9ada-3c9aa934d199'

In [14]:
# role ฝั่งหน่วยงานต้องมี ORGANIZATION_ID เสมอ — ระบบไม่สร้างหน่วยงานเปล่าให้แล้ว
# ไม่ส่ง organizationId มาจะได้ 400 ไม่ใช่คำเชิญที่ไม่ผูกหน่วยงานแบบเดิม
# เซลล์ 0.6 เป็นคนตั้งค่านี้ให้ — เปิดบรรทัดล่างเฉพาะตอนอยากผูกหน่วยงานที่มีอยู่แล้ว
# ORGANIZATION_ID = '17901807-5754-4664-9411-5d7159642524'
if ROLE in ORG_SCOPED_ROLES and not ORGANIZATION_ID:
    raise SystemExit(
        f"หยุด: role {ROLE} เป็น role ระดับหน่วยงาน จึงต้องมี ORGANIZATION_ID เสมอ — "
        "กลับไปรันเซลล์ 0.6 ให้สำเร็จก่อน (ดู error ด้านบน เช่นชื่ออำเภอ/ตำบลไม่ตรง) "
        "หรือใส่ id ของหน่วยงานที่มีอยู่แล้วลงในตัวแปรนี้เอง"
    )

# ชื่อจริงของผู้ถูกเชิญ — บังคับทุก role แล้ว เอกสาร A0–A3 ใช้ชื่อนี้ ไม่ใช่ชื่อที่แสดง
# คำนำหน้าไม่บังคับ แต่ควรใส่ เพราะ ThaiD ไม่ส่งคำนำหน้ามาให้ (claim title อยู่นอก scope)
INVITE_PREFIX = "นาย"
INVITE_FIRSTNAME = "ทดสอบ"
INVITE_LASTNAME = "ระบบดี"

payload = {"email": INVITE_EMAIL, "role": ROLE, "cid": CID,
           "prefixTh": INVITE_PREFIX, "firstnameTh": INVITE_FIRSTNAME,
           "lastnameTh": INVITE_LASTNAME}
if ROLE in ORG_SCOPED_ROLES:
    payload["organizationId"] = ORGANIZATION_ID

response, invitation = call(
    "POST",
    "/api/admin/invitations",
    headers={"x-admin-token": ADMIN_TOKEN},
    json=payload,
)

if response.status_code == 201:
    print("\n📧 เปิดกล่องจดหมายของ", INVITE_EMAIL, "— หัวข้อ 'คำเชิญเข้าใช้งาน Government Datahub Platform'")
    print("   บัญชีถูกสร้างเป็น PENDING แล้ว · userAccountId =", invitation["userAccountId"])
    if ROLE in ORG_SCOPED_ROLES and not ORGANIZATION_ID:
        print("   ไม่ได้ระบุหน่วยงาน — ระบบเตรียมหน่วยงานเปล่าไว้ให้แล้ว organizationId =",
              invitation["organizationId"])
        print("   เขาจะได้กรอกชื่อหน่วยงานเองในฟอร์ม Journey B")
elif response.status_code == 409:
    print("\nอีเมลนี้มีบัญชีที่ ACTIVE อยู่แล้ว — เปลี่ยนเป็น +demo2 แล้วรันใหม่")


POST /api/admin/invitations  →  201 Created

{
  "activationKeyId": "c9fc4499-4f44-4bcc-8101-ac3ce153ea22",
  "userAccountId": "89e71082-7d90-4229-af89-419c77dd8920",
  "email": "thammasorn.han+demo21@gmail.com",
  "role": "ORGANIZATION_USER",
  "roleLabel": "ผู้ดำเนินการของหน่วยงาน",
  "organizationId": "759f8e08-07e9-4ae6-9ada-3c9aa934d199",
  "expiresAt": "2026-09-05T16:25:11.786Z"
}

📧 เปิดกล่องจดหมายของ thammasorn.han+demo21@gmail.com — หัวข้อ 'คำเชิญเข้าใช้งาน Government Datahub Platform'
   บัญชีถูกสร้างเป็น PENDING แล้ว · userAccountId = 89e71082-7d90-4229-af89-419c77dd8920


In [15]:
# หน่วยงานที่คำเชิญนี้ผูกไว้ — จากคำตอบของเซลล์ก่อนหน้า
if invitation:
    print("organizationId ของคำเชิญ :", invitation.get("organizationId") or "(ไม่ได้ผูก)")

organizationId ของคำเชิญ : 759f8e08-07e9-4ae6-9ada-3c9aa934d199


In [ ]:
Print Stop หยุดตรงนี้

## 2. เปิดอีเมล แล้วเอาลิงก์มาวาง

ในอีเมลมีปุ่ม **เปิดใช้งานบัญชี** ชี้ไปที่ `<APP_URL>/activate?token=…`
คลิกขวาที่ปุ่ม → คัดลอกลิงก์ แล้ววางในเซลล์ถัดไป (จะวางทั้งลิงก์หรือเฉพาะ token ก็ได้)

> ถ้า checkout อยู่โหมดพิมพ์ลง log ให้ข้ามไปเซลล์ **หาลิงก์จาก log** ท้ายไฟล์นี้


In [ ]:
PASTED = ""  # ← วางลิงก์จากอีเมลตรงนี้

TOKEN = parse_qs(urlparse(PASTED).query).get("token", [PASTED])[0].strip()
print("token :", TOKEN[:12] + "…" if TOKEN else "(ยังไม่ได้วาง)")

call("GET", "/api/auth/invitation", params={"token": TOKEN})

## 3. ยืนยันตัวตนด้วย ThaiD

หัวใจของ §2.4: ระบบเทียบเลขบัตรที่ ThaiD ส่งกลับกับ `user_account.cid` ที่เจ้าหน้าที่บันทึกไว้
ตอนสร้างบัญชี **ไม่ตรง = ยกเลิก activation key ทิ้งทันที** ลิงก์เดิมใช้ต่อไม่ได้ ต้องขอใหม่

เซลล์ถัดไปยิง `POST /api/auth/thaid/start` แล้วคืน `authorizeUrl` มาให้ เปิดลิงก์นั้นในเบราว์เซอร์
เพื่อยืนยัน เสร็จแล้วเบราว์เซอร์จะถูกส่งกลับไปที่ `<APP_URL>/auth/callback/thaid?code=…&state=…`
ซึ่งหน้าเว็บทำต่อให้เสร็จ (บน sandbox หน้าของกรมการปกครองแก้เลขบัตรได้ ใช้สาธิตกรณี
"เลขไม่ตรง" ได้ตรงนั้น)

ถ้าได้ `501 not_configured` แปลว่า checkout นั้นไม่ได้ตั้ง `THAID_CLIENT_ID/SECRET` —
เติม credentials ใน `.env` ก่อน แล้วรีสตาร์ต backend

In [ ]:
response, start = call("POST", "/api/auth/thaid/start", json={"purpose": "activate", "token": TOKEN})

if response.status_code == 200:
    print("\n🔗 เปิดลิงก์นี้ในเบราว์เซอร์เพื่อยืนยันตัวตนด้วย ThaiD:")
    print("  ", start["authorizeUrl"])
    print("\n   ยืนยันเสร็จแล้วเบราว์เซอร์จะพากลับมาที่หน้า 'สร้างบัญชีผู้ใช้' ของระบบ")
    print("   จะตั้งรหัสผ่านบนหน้าเว็บต่อเลยก็ได้ หรือกลับมารันเซลล์ถัดไปใน notebook ก็ได้")
elif response.status_code == 501:
    print("\n⚠️  ยืนยันตัวตนไม่ได้ — checkout นี้ไม่ได้ตั้ง THAID_CLIENT_ID/SECRET")

## 4. ตั้งรหัสผ่านและเปิดใช้งานบัญชี

`POST /api/auth/activate` จะสำเร็จก็ต่อเมื่อมี "ใบเสร็จ" การยืนยันตัวตนของ activation key
ใบเดียวกันที่ยังไม่หมดอายุ (30 นาที) — ไม่มีก็ตอบ `409 identity_required` ให้กลับไปทำขั้นที่ 3

ขั้นนี้ทำสามอย่างใน transaction เดียวตามลำดับใน sheet `activation_key`:
บัญชีเปลี่ยนเป็น `ACTIVE` · สร้าง `user_role_assignment` · ปิด key เป็น `USED`
แล้วเซิร์ฟเวอร์ตั้ง session cookie ให้ — `http` เก็บ cookie นั้นไว้ต่อ

- เบอร์โทรต้องขึ้นต้น `0` และมี 9–10 หลัก
- รหัสผ่านอย่างน้อย 8 ตัว มีทั้งตัวอักษรและตัวเลข

ลองกรอกผิดดูได้ — ระบบตอบเป็น `{"error":"validation","fields":{…}}` ระบุทีละช่อง


In [ ]:
# ใช้ชื่อชุดเดียวกับที่เชิญไป — ฟอร์มจริงจะถูก prefill ด้วยค่าเหล่านี้อยู่แล้ว
# (ThaiD มาก่อน ถ้าไม่ได้ชื่อจาก ThaiD จึงตกมาใช้ค่าจากคำเชิญ)
PROFILE = {
    "prefix": INVITE_PREFIX,
    "firstName": INVITE_FIRSTNAME,
    "lastName": INVITE_LASTNAME,
    "phone": "0812345678",
    "password": "bdi12345",
}

response, _ = call("POST", "/api/auth/activate", json={"token": TOKEN, **PROFILE})

if response.status_code == 200:
    print("\ncookie ที่ได้รับ :", ", ".join(http.cookies.keys()) or "(ไม่มี)")
elif response.status_code == 409:
    print("\nยังไม่ได้ยืนยันตัวตนด้วย ThaiD (หรือใบเสร็จหมดอายุ) — กลับไปรันขั้นที่ 3")


## 5. ตรวจผล

บัญชีใหม่เข้าใช้งานได้แล้ว — เข้า `APP_URL` ด้วยอีเมลนี้กับรหัสผ่านที่ตั้งไว้
เพื่อเดินต่อเป็น Journey B (สร้างหน่วยงาน) ได้ทันที

`roles` ใน `/me` อ่านจาก `iam.user_role_assignment` สด ๆ ทุก request ไม่ได้มาจาก cookie

In [ ]:
call("GET", "/api/auth/me")

_, listing = call("GET", "/api/admin/invitations", headers={"x-admin-token": ADMIN_TOKEN})
if listing:
    mine = [i for i in listing["invitations"] if i["userAccount"]["email"] == INVITE_EMAIL]
    print("\nactivation key ของอีเมลนี้ (ล่าสุดอยู่บน) :")
    for item in mine:
        org = item["organization"]["nameTh"] if item.get("organization") else "—"
        print(f"  {item['status']:8} {item['role']['code']:24} {org:40} {item['createdAt']}")
    print("\nUSED = ใช้ไปแล้ว · REVOKED = ถูกแทนที่ด้วยคีย์ใหม่ · EXPIRED = หมดอายุ · ISSUED = ยังใช้ได้")

print("\nเข้าสู่ระบบต่อได้ที่", APP_URL, "ด้วย", INVITE_EMAIL, "/", PROFILE["password"])

## 6. ทางลัด — สร้างทีมฝั่ง BDI ทีเดียว (officer · specialist · approver)

Journey B/C ต้องมีเจ้าหน้าที่ BDI คอยตรวจและอนุมัติ เซลล์นี้เชิญทั้งสาม role รวดเดียว
โดยใช้อีเมลของคุณกล่องเดียว แล้วเติม `+officer1` `+specialist1` `+approver1`
(plus-addressing ของ Gmail — ทุกกล่องเข้าอีเมลเดียวกัน) · role ฝั่ง BDI ไม่ต้องระบุหน่วยงาน
ระบบผูกกับหน่วยงาน BDI ให้เอง

- `BDI_OFFICER` — เจ้าหน้าที่ตรวจสอบเบื้องต้น (ด่านแรกของทั้งสอง Journey)
- `BDI_DATASET_SPECIALIST` — ผู้เชี่ยวชาญตรวจชุดข้อมูล (เฉพาะ Journey C)
- `BDI_FINAL_APPROVER` — ผู้อนุมัติขั้นสุดท้าย

เลขบัตรของแต่ละบัญชีถูกสร้างให้อัตโนมัติ (ผ่าน checksum) จาก `CID_SEED` — ถ้าชนของเดิม
(`cid_exists`) ให้บวก `CID_SEED` แล้วรันใหม่ · แต่ละบัญชียังต้องเปิดใช้งานเอง (ทำขั้น 2–4
โดยชี้ `INVITE_EMAIL` ไปที่อีเมล `+tag` นั้น)

In [ ]:
# ── ใส่อีเมลของคุณตรงนี้ · ดีฟอลต์ดึงฐานจาก INVITE_EMAIL (ตัด +tag เดิมออกให้) ──
def _base_email(email: str) -> str:
    local, domain = email.split("@", 1)
    return local.split("+")[0] + "@" + domain


BASE_EMAIL = _base_email(INVITE_EMAIL)   # เช่น "wanisara.harn@gmail.com"

# (แท็กที่เติมหลัง + , รหัส role)  — แก้/เพิ่มได้ เช่นอยากได้ BDI_LEGAL_OFFICER ก็เติมบรรทัด
BDI_TEAM = [
    ("officer1",    "BDI_OFFICER"),
    ("specialist1", "BDI_DATASET_SPECIALIST"),
    ("approver1",   "BDI_FINAL_APPROVER"),
]
CID_SEED = 700000000000   # ฐานเลขบัตร 12 หลัก — ถ้าชนของเดิม (cid_exists) บวกค่านี้แล้วรันใหม่


def _plus(email: str, tag: str) -> str:
    local, domain = email.split("@", 1)
    return f"{local}+{tag}@{domain}"


def _thai_cid(n: int) -> str:
    """สร้างเลขบัตร 13 หลักที่ผ่าน checksum จากเลขฐาน 12 หลัก (อัลกอริทึม modulo-11)"""
    body = f"{n:012d}"
    check = (11 - sum(int(body[i]) * (13 - i) for i in range(12)) % 11) % 10
    return body + str(check)


created = []
for offset, (tag, role) in enumerate(BDI_TEAM):
    email = _plus(BASE_EMAIL, tag)
    cid = _thai_cid(CID_SEED + offset)
    print(f"\n── {role}  →  {email}  (cid {cid}) ──")
    response, body = call(
        "POST",
        "/api/admin/invitations",
        headers={"x-admin-token": ADMIN_TOKEN},
        json={"email": email, "role": role, "cid": cid},   # role ฝั่ง BDI ไม่ต้องมี organizationId
    )
    if response.status_code == 201:
        created.append((role, email))
    elif response.status_code == 409:
        # อีเมลมีบัญชี ACTIVE แล้ว หรือเลขบัตรซ้ำ — ข้ามไป ไม่ใช่ error ของทั้งชุด
        print(f"   ข้าม — {body.get('error')}: {body.get('message')}")

print(f"\nสรุป: เชิญสำเร็จ {len(created)}/{len(BDI_TEAM)} บัญชี")
if created:
    print("เปิดอีเมลแต่ละกล่อง (หัวข้อ 'คำเชิญเข้าใช้งาน…') แล้วเปิดใช้งานที่", APP_URL)
    print("เปิดใช้งานทีละคน: ตั้ง INVITE_EMAIL เป็นอีเมล +tag นั้น แล้วทำขั้น 2–4")

---

## หาลิงก์และ OTP จาก log (เมื่อ checkout ไม่ได้ส่งอีเมลจริง)

เมื่อ `SMTP_USER` ว่าง mailer จะพิมพ์เนื้ออีเมล ลิงก์ และ OTP (ของการเข้าสู่ระบบ) ลง stdout แทนการส่ง

เซลล์นี้แสดงเฉพาะ**รอบล่าสุดของอีเมลที่กำลังเชิญ** — log เก็บทุกรอบไว้ปนกัน
การหยิบลิงก์หรือ OTP ของรอบก่อนมาใช้จะได้ `400` โดยไม่มีอะไรบอกว่าเพราะอะไร


In [ ]:
logs = subprocess.run(
    ["docker", "compose", "logs", "backend", "--tail", "400"],
    cwd=CHECKOUT, capture_output=True, text=True,
).stdout.splitlines()

# ตัดเอาเฉพาะตั้งแต่ครั้งสุดท้ายที่อีเมลนี้ถูกส่งถึง
starts = [i for i, line in enumerate(logs) if INVITE_EMAIL in line]
recent = logs[starts[-1]:] if starts else []

if not recent:
    print(f"ยังไม่พบ {INVITE_EMAIL} ใน log 400 บรรทัดล่าสุด — เชิญไปแล้วหรือยัง?")
else:
    for line in recent:
        if "activate?token=" in line or "รหัส OTP" in line or INVITE_EMAIL in line:
            print(line)
    print("\nถ้ามีทั้งลิงก์และ OTP ให้ใช้ของบรรทัดล่างสุดเสมอ — บนสุดคือรอบก่อนหน้า")


## เริ่มรอบใหม่

เปลี่ยน `INVITE_EMAIL` เป็น `+demo2`, `+demo3` … แล้วรันตั้งแต่เซลล์ที่ 1 ใหม่
ถ้าต้องการยกเลิกคีย์ที่ยังใช้ได้โดยไม่ออกใบใหม่ ใช้เซลล์ล่างนี้

In [ ]:
_, listing = call("GET", "/api/admin/invitations", headers={"x-admin-token": ADMIN_TOKEN})
issued = [
    i for i in listing["invitations"]
    if i["userAccount"]["email"] == INVITE_EMAIL and i["status"] == "ISSUED"
]

for item in issued:
    call(
        "POST",
        f"/api/admin/invitations/{item['id']}/revoke",
        headers={"x-admin-token": ADMIN_TOKEN},
        json={"reason": "ยกเลิกจาก notebook"},
    )

if not issued:
    print("ไม่มี activation key ที่ยังใช้ได้ของอีเมลนี้")